# 03 - Load PostgreSQL and Build OLAP

This notebook loads raw and processed CSV files into PostgreSQL schemas, then builds OLAP aggregate tables and dashboard-ready CSV exports.

## Setup Database Connection

This section prepares the PostgreSQL connection using values from `.env`. The notebook supports either individual DB fields such as `DB_HOST` and `DB_USER`, or a full `DATABASE_URL`.

**Input:** `.env` database configuration.  
**Output:** SQLAlchemy database engine.

In [1]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'output'
VALIDATION_DIR = PROJECT_ROOT / 'data' / 'validation'
SQL_DIR = PROJECT_ROOT / 'sql'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

def get_database_url():
    load_dotenv(PROJECT_ROOT / '.env')
    if os.getenv('DATABASE_URL'):
        return os.getenv('DATABASE_URL')
    return f"postgresql+psycopg2://{os.getenv('DB_USER','postgres')}:{os.getenv('DB_PASSWORD','postgres')}@{os.getenv('DB_HOST','localhost')}:{os.getenv('DB_PORT','5432')}/{os.getenv('DB_NAME','erajaya_dw')}"
engine = create_engine(get_database_url(), pool_pre_ping=True)

## Create Database Schemas

This section executes the SQL schema files for OLTP, staging, data warehouse, and OLAP layers. Running this step resets the target tables so the pipeline can be rerun consistently.

**Input:** SQL files in `sql/`.  
**Output:** PostgreSQL schemas `oltp`, `staging`, `dw`, and `olap`.

In [2]:
def execute_sql_file(filename):
    sql_text = (SQL_DIR / filename).read_text(encoding='utf-8')
    with engine.begin() as conn:
        raw = conn.connection.driver_connection
        with raw.cursor() as cursor:
            cursor.execute(sql_text)
try:
    for filename in ['01_oltp_schema.sql', '02_staging_schema.sql', '03_dw_schema.sql', '04_olap_schema.sql']:
        execute_sql_file(filename)
        print('Executed', filename)
except SQLAlchemyError as exc:
    raise SystemExit('PostgreSQL is not reachable or .env is invalid. Start PostgreSQL, create erajaya_dw, and check .env. Original error: ' + str(exc))

Executed 01_oltp_schema.sql
Executed 02_staging_schema.sql
Executed 03_dw_schema.sql
Executed 04_olap_schema.sql


## Load Raw Data into OLTP

This section loads raw source CSV files into normalized OLTP tables. It also derives the channel table from transaction data and ensures missing customers are mapped to `CUST-GUEST`.

**Input:** CSV files in `data/raw/`.  
**Output:** populated tables in the `oltp` schema.

In [3]:
def read_raw(name):
    return pd.read_csv(RAW_DIR / name)
def clean_dates(df, columns):
    for column in columns:
        df[column] = pd.to_datetime(df[column], errors='coerce').dt.date
        df[column] = df[column].where(pd.notna(df[column]), None)
    return df
customers = clean_dates(read_raw('customers.csv'), ['birth_date','registered_date'])
stores = clean_dates(read_raw('stores.csv'), ['open_date'])
products = clean_dates(read_raw('products.csv'), ['launch_date'])
promotions = clean_dates(read_raw('promotions.csv'), ['start_date','end_date'])
transactions = read_raw('sales_transactions.csv'); details = read_raw('sales_details.csv'); payments = read_raw('payments.csv')
inventory = clean_dates(read_raw('inventory.csv'), ['snapshot_date'])
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], errors='coerce')
transactions['customer_id'] = transactions['customer_id'].fillna('').replace('', 'CUST-GUEST')
transactions['salesperson_id'] = transactions['salesperson_id'].fillna('')
channels = transactions[['channel_id','channel_name','channel_type']].drop_duplicates().sort_values('channel_id')
transactions = transactions[['transaction_id','transaction_date','customer_id','store_id','channel_id','payment_id','salesperson_id']]
oltp_frames = {'tb_customer':customers,'tb_channel':channels,'tb_store':stores,'tb_product':products,'tb_promotion':promotions,'tb_payment':payments,'tb_sales_transaction':transactions,'tb_sales_detail':details,'tb_inventory':inventory}
with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE oltp.tb_inventory, oltp.tb_sales_detail, oltp.tb_sales_transaction, oltp.tb_payment, oltp.tb_promotion, oltp.tb_product, oltp.tb_store, oltp.tb_channel, oltp.tb_customer RESTART IDENTITY CASCADE'))
for table, df in oltp_frames.items():
    df.to_sql(table, engine, schema='oltp', if_exists='append', index=False, method='multi', chunksize=1000)
    print(f'Loaded oltp.{table}: {len(df):,} rows')

Loaded oltp.tb_customer: 1,000 rows
Loaded oltp.tb_channel: 3 rows
Loaded oltp.tb_store: 25 rows
Loaded oltp.tb_product: 100 rows
Loaded oltp.tb_promotion: 21 rows
Loaded oltp.tb_payment: 2,000 rows


Loaded oltp.tb_sales_transaction: 2,000 rows


Loaded oltp.tb_sales_detail: 4,688 rows


Loaded oltp.tb_inventory: 2,500 rows


## Extract OLTP Data into Staging

This section copies OLTP data into staging tables. The staging layer acts as a controlled extraction area before data is loaded into the warehouse layer.

**Input:** populated `oltp` tables.  
**Output:** populated `staging.stg_*` tables.

In [4]:
pairs = [('stg_customer','tb_customer'),('stg_channel','tb_channel'),('stg_store','tb_store'),('stg_product','tb_product'),('stg_promotion','tb_promotion'),('stg_payment','tb_payment'),('stg_sales_transaction','tb_sales_transaction'),('stg_sales_detail','tb_sales_detail'),('stg_inventory','tb_inventory')]
with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE staging.stg_inventory, staging.stg_sales_detail, staging.stg_sales_transaction, staging.stg_payment, staging.stg_promotion, staging.stg_product, staging.stg_store, staging.stg_channel, staging.stg_customer RESTART IDENTITY CASCADE'))
    for stg, src in pairs:
        conn.execute(text(f'INSERT INTO staging.{stg} SELECT * FROM oltp.{src}'))
        print(f"Loaded staging.{stg}: {conn.execute(text(f'SELECT COUNT(*) FROM staging.{stg}')).scalar_one():,} rows")

Loaded staging.stg_customer: 1,000 rows
Loaded staging.stg_channel: 3 rows
Loaded staging.stg_store: 25 rows
Loaded staging.stg_product: 100 rows
Loaded staging.stg_promotion: 21 rows
Loaded staging.stg_payment: 2,000 rows
Loaded staging.stg_sales_transaction: 2,000 rows
Loaded staging.stg_sales_detail: 4,688 rows
Loaded staging.stg_inventory: 2,500 rows


## Load Dimension and Fact Tables into the Warehouse

This section reads processed dimension and fact CSV files from `data/processed/` and loads them into the `dw` schema. These processed files are produced by Notebook 02.

**Input:** processed CSV files such as `dim_product.csv` and `fact_sales.csv`.  
**Output:** populated `dw.dim_*` and `dw.fact_*` tables.

In [5]:
def read_processed(name):
    return pd.read_csv(PROCESSED_DIR / name)
dw_frames = {'dim_time':read_processed('dim_time.csv'),'dim_customer':read_processed('dim_customer.csv'),'dim_product':read_processed('dim_product.csv'),'dim_store':read_processed('dim_store.csv'),'dim_channel':read_processed('dim_channel.csv'),'dim_payment':read_processed('dim_payment.csv'),'dim_promotion':read_processed('dim_promotion.csv'),'fact_sales':read_processed('fact_sales.csv'),'fact_inventory_snapshot':read_processed('fact_inventory_snapshot.csv')}
for table in ['dim_time','dim_customer','dim_product','dim_store','dim_promotion']:
    for col in dw_frames[table].columns:
        if col.endswith('date') or col == 'full_date':
            dw_frames[table][col] = pd.to_datetime(dw_frames[table][col], errors='coerce').dt.date
with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE dw.fact_inventory_snapshot, dw.fact_sales, dw.dim_promotion, dw.dim_payment, dw.dim_channel, dw.dim_store, dw.dim_product, dw.dim_customer, dw.dim_time RESTART IDENTITY CASCADE'))
for table, df in dw_frames.items():
    df.to_sql(table, engine, schema='dw', if_exists='append', index=False, method='multi', chunksize=1000)
    print(f'Loaded dw.{table}: {len(df):,} rows')

Loaded dw.dim_time: 513 rows


Loaded dw.dim_customer: 1,000 rows
Loaded dw.dim_product: 100 rows
Loaded dw.dim_store: 25 rows
Loaded dw.dim_channel: 3 rows


Loaded dw.dim_payment: 2,000 rows
Loaded dw.dim_promotion: 21 rows


Loaded dw.fact_sales: 4,688 rows
Loaded dw.fact_inventory_snapshot: 2,500 rows


## Build OLAP Tables and Export Dashboard CSV Files

This section builds analytical aggregate tables from the warehouse facts and dimensions. The same result sets are also exported to `output/` so the dashboard team can use them without directly connecting to PostgreSQL.

**Input:** populated `dw` schema.  
**Output:** `olap.agg_*`, `olap.mart_sales_overview`, and dashboard-ready CSV files in `output/`.

In [6]:
AGG_QUERIES = {
    'agg_monthly_sales': """SELECT t.year, t.month, SUM(f.net_sales) AS total_sales, SUM(f.net_profit) AS total_profit, SUM(f.quantity_sold) AS total_quantity, COUNT(DISTINCT f.transaction_id) AS total_transactions FROM dw.fact_sales f JOIN dw.dim_time t ON f.time_key = t.time_key GROUP BY t.year, t.month ORDER BY t.year, t.month""",
    'agg_sales_by_channel': """SELECT c.channel_name, c.channel_type, SUM(f.net_sales) AS total_sales, SUM(f.net_profit) AS total_profit, COUNT(DISTINCT f.transaction_id) AS total_transactions FROM dw.fact_sales f JOIN dw.dim_channel c ON f.channel_key = c.channel_key GROUP BY c.channel_name, c.channel_type ORDER BY total_sales DESC""",
    'agg_sales_by_region': """SELECT s.province, s.city, SUM(f.net_sales) AS total_sales, SUM(f.net_profit) AS total_profit, COUNT(DISTINCT f.transaction_id) AS total_transactions FROM dw.fact_sales f JOIN dw.dim_store s ON f.store_key = s.store_key GROUP BY s.province, s.city ORDER BY total_sales DESC""",
    'agg_top_products': """SELECT p.product_name, p.brand, p.category, SUM(f.quantity_sold) AS total_quantity, SUM(f.net_sales) AS total_sales, SUM(f.net_profit) AS total_profit FROM dw.fact_sales f JOIN dw.dim_product p ON f.product_key = p.product_key GROUP BY p.product_name, p.brand, p.category ORDER BY total_sales DESC LIMIT 50""",
    'agg_inventory_stockout': """SELECT s.store_name, s.city, p.product_name, p.brand, i.stock_quantity, i.reorder_level, i.stock_status FROM dw.fact_inventory_snapshot i JOIN dw.dim_store s ON i.store_key = s.store_key JOIN dw.dim_product p ON i.product_key = p.product_key WHERE i.stock_status IN ('Stockout', 'Low Stock') ORDER BY i.stock_quantity ASC, s.store_name, p.product_name""",
    'mart_sales_overview': """SELECT f.transaction_id, f.detail_id, t.full_date, t.year, t.month, c.customer_id, c.customer_name, c.loyalty_tier, c.customer_segment, p.product_id, p.product_name, p.brand, p.category, s.store_name, s.store_type, s.province, s.city, ch.channel_name, ch.channel_type, pay.payment_method, pay.payment_status, promo.promotion_name, promo.promotion_type, f.quantity_sold, f.unit_price, f.gross_sales, f.discount_amount, f.net_sales, f.cost_amount, f.net_profit FROM dw.fact_sales f JOIN dw.dim_time t ON f.time_key = t.time_key JOIN dw.dim_customer c ON f.customer_key = c.customer_key JOIN dw.dim_product p ON f.product_key = p.product_key JOIN dw.dim_store s ON f.store_key = s.store_key JOIN dw.dim_channel ch ON f.channel_key = ch.channel_key JOIN dw.dim_payment pay ON f.payment_key = pay.payment_key JOIN dw.dim_promotion promo ON f.promotion_key = promo.promotion_key""",
}
with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE olap.mart_sales_overview, olap.agg_inventory_stockout, olap.agg_top_products, olap.agg_sales_by_region, olap.agg_sales_by_channel, olap.agg_monthly_sales'))
for table, query in AGG_QUERIES.items():
    df = pd.read_sql_query(text(query), engine)
    df.to_sql(table, engine, schema='olap', if_exists='append', index=False, method='multi', chunksize=1000)
    df.to_csv(OUTPUT_DIR / f'{table}.csv', index=False)
    print(f'Built olap.{table}: {len(df):,} rows')

Built olap.agg_monthly_sales: 17 rows
Built olap.agg_sales_by_channel: 3 rows
Built olap.agg_sales_by_region: 13 rows
Built olap.agg_top_products: 50 rows
Built olap.agg_inventory_stockout: 451 rows


Built olap.mart_sales_overview: 4,688 rows


## Load Validation Outputs

Create validation CSV files for the PostgreSQL loading step. These files confirm which tables were loaded, whether key warehouse columns exist, and the final row count in selected OLTP, staging, DW, and OLAP tables.

**Output:** data/validation/load_results.csv, data/validation/load_column_validation.csv, and data/validation/load_final_counts.csv.


In [7]:
expected_columns = {
    'dw.dim_time': ['time_key','full_date','year','quarter','month','month_name','day','day_of_week','is_weekend'],
    'dw.dim_customer': ['customer_key','customer_id','customer_name','gender','birth_date','loyalty_tier','customer_segment','province','city','registered_date'],
    'dw.dim_product': ['product_key','product_id','product_name','brand','category','subcategory','unit_price','cost_price','launch_date','is_active'],
    'dw.dim_store': ['store_key','store_id','store_name','store_type','province','city','region','open_date'],
    'dw.dim_channel': ['channel_key','channel_id','channel_name','channel_type'],
    'dw.dim_payment': ['payment_key','payment_id','payment_method','payment_status','payment_provider'],
    'dw.dim_promotion': ['promotion_key','promotion_id','promotion_name','promotion_type','discount_rate','start_date','end_date'],
    'dw.fact_sales': ['sales_key','transaction_id','detail_id','time_key','customer_key','product_key','store_key','channel_key','payment_key','promotion_key','quantity_sold','unit_price','gross_sales','discount_amount','net_sales','cost_amount','net_profit'],
    'dw.fact_inventory_snapshot': ['inventory_key','time_key','product_key','store_key','stock_quantity','reorder_level','stock_status'],
}

load_column_rows = []
load_count_rows = []

with engine.connect() as conn:
    for schema_table, expected in expected_columns.items():
        schema_name, table_name = schema_table.split('.')
        actual = [row[0] for row in conn.execute(text("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = :schema_name AND table_name = :table_name
            ORDER BY ordinal_position
        """), {'schema_name': schema_name, 'table_name': table_name}).fetchall()]
        missing = [col for col in expected if col not in actual]
        extra = [col for col in actual if col not in expected]
        load_column_rows.append({
            'schema_table': schema_table,
            'expected_column_count': len(expected),
            'actual_column_count': len(actual),
            'missing_columns': ', '.join(missing),
            'extra_columns': ', '.join(extra),
            'is_column_valid': len(missing) == 0,
        })

    count_tables = [
        'oltp.tb_customer', 'oltp.tb_product', 'oltp.tb_store', 'oltp.tb_sales_transaction', 'oltp.tb_sales_detail',
        'staging.stg_customer', 'staging.stg_product', 'staging.stg_sales_transaction', 'staging.stg_sales_detail',
        'dw.dim_customer', 'dw.dim_product', 'dw.dim_store', 'dw.fact_sales', 'dw.fact_inventory_snapshot',
        'olap.agg_monthly_sales', 'olap.agg_sales_by_channel', 'olap.agg_sales_by_region', 'olap.agg_top_products',
        'olap.agg_inventory_stockout', 'olap.mart_sales_overview'
    ]
    for schema_table in count_tables:
        row_count = conn.execute(text(f'SELECT COUNT(*) FROM {schema_table}')).scalar_one()
        load_count_rows.append({'schema_table': schema_table, 'row_count': row_count})

if 'load_results' not in globals() or not load_results:
    load_results = [
        {'layer': item['schema_table'].split('.')[0], 'table_name': item['schema_table'].split('.')[1], 'rows_loaded': item['row_count'], 'status': 'COUNT_ONLY'}
        for item in load_count_rows
    ]

pd.DataFrame(load_results).to_csv(VALIDATION_DIR / 'load_results.csv', index=False)
pd.DataFrame(load_column_rows).to_csv(VALIDATION_DIR / 'load_column_validation.csv', index=False)
pd.DataFrame(load_count_rows).to_csv(VALIDATION_DIR / 'load_final_counts.csv', index=False)

print('Load validation files written to:', VALIDATION_DIR)
pd.DataFrame(load_count_rows)

Load validation files written to: D:\Semester 8\Data Warehouse\erajaya-data-warehouse\data\validation


,schema_table,row_count
0,oltp.tb_customer,1000
1,oltp.tb_product,100
2,oltp.tb_store,25
3,oltp.tb_sales_transaction,2000
4,oltp.tb_sales_detail,4688
5,staging.stg_customer,1000
6,staging.stg_product,100
7,staging.stg_sales_transaction,2000
8,staging.stg_sales_detail,4688
9,dw.dim_customer,1000
